Prueba

In [1]:
pip install hyperopt

Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from hyperopt import hp, fmin, tpe, Trials, STATUS_OK
from keras.models import Sequential
from keras.layers import Dense, Dropout, GRU, InputLayer, Flatten
from keras.callbacks import EarlyStopping, ModelCheckpoint
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
import os
np.random.seed(42)


LEER DATASET

In [3]:
import pandas as pd

datos = pd.read_csv("C:\\Users\\wamt1\\OneDrive\\Escritorio\\colab\\SNconsumptionFinal.csv")

datos['date'] = pd.to_datetime(datos['date'])

# Se establece la columna date como index
datos.set_index('date', inplace=True)


In [4]:
datos.head()

,temp,zone1,zone2,zone3,hour
date,,,,,
2017-01-01 00:00:00,6.559,34055.69620,16128.87538,20240.96386,0
2017-01-01 00:10:00,6.414,29814.68354,19375.07599,20131.08434,0
2017-01-01 00:20:00,6.313,29128.10127,19006.68693,19668.43373,0
2017-01-01 00:30:00,6.121,28228.86076,18361.09422,18899.27711,0
2017-01-01 00:40:00,5.921,27335.69620,17872.34043,18442.40964,0


In [5]:
#Se verifica que el index está en formato de dato "datetime64"
print("El index del dataframe input es un tipo de dato: ", datos.index.dtype)


El index del dataframe input es un tipo de dato:  datetime64[ns]


Se divide el dataset

In [6]:
# Dividir el conjunto de datos en entrenamiento y prueba
train, test = train_test_split(datos, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
test, val = train_test_split(test, test_size=0.33, shuffle=False)

print("Las dimensiones de train son: ", train.shape)
print("Las dimensiones de test son: ", test.shape)
print("Las dimensiones de val son: ", val.shape)

Las dimensiones de train son:  (36691, 5)
Las dimensiones de test son:  (10535, 5)
Las dimensiones de val son:  (5190, 5)


Se normalizan los datos

In [7]:
from sklearn.preprocessing import StandardScaler


# Normalizar solo con los datos de entrenamiento
scaler = StandardScaler()
train = scaler.fit_transform(train)

# Aplicar la transformación a test y val usando los parámetros de train
test = scaler.transform(test)
val = scaler.transform(val)

train = pd.DataFrame(train, columns=datos.columns)
test = pd.DataFrame(test, columns=datos.columns)
val = pd.DataFrame(val, columns=datos.columns)


Se unen los datos nuevamente, ahora normalizados, en un único conjunto.

In [8]:
datosNormalizados = pd.concat([train, test, val])

datosNormalizados.index = datos.index


In [9]:
datosNormalizados.shape

(52416, 5)

In [10]:
datosNormalizados.head(10)


,temp,zone1,zone2,zone3,hour
date,,,,,
2017-01-01 00:00:00,-2.051356,0.151445,-0.851669,0.033500,-1.660858
2017-01-01 00:10:00,-2.074813,-0.433079,-0.211097,0.016502,-1.660858
2017-01-01 00:20:00,-2.091152,-0.527709,-0.283791,-0.055068,-1.660858
2017-01-01 00:30:00,-2.122213,-0.651648,-0.411186,-0.174054,-1.660858
2017-01-01 00:40:00,-2.154568,-0.774750,-0.507632,-0.244729,-1.660858
2017-01-01 00:50:00,-2.165569,-0.872729,-0.597600,-0.293039,-1.660858
2017-01-01 01:00:00,-2.199865,-0.958984,-0.681090,-0.321667,-1.516340
2017-01-01 01:10:00,-2.223323,-1.035189,-0.746587,-0.396816,-1.516340
2017-01-01 01:20:00,-2.193880,-1.127306,-0.832236,-0.463913,-1.516340


Espacio de búsqueda

In [11]:
space = {
    'layers': hp.quniform('layers', 1, 4, 1), # Cantidad de capas GRU
    'units': hp.choice('units', [2 ** i for i in range(3, 8)]),  # Número de unidades GRU
    'activation': hp.choice('activation', ['tanh', 'sigmoid', 'relu', 'linear']),
    'dropout': hp.quniform('dropout', 0, 0.5, 0.1),  # Dropout para regularización
    'learning_rate': hp.loguniform('learning_rate', np.log(0.000001), np.log(0.01)),  # Tasa de aprendizaje
    'epochs': hp.choice('epochs', [2 ** i for i in range(3, 9)]),  # Número de épocas de entrenamiento
    'batch': hp.choice('batch',[2 ** i for i in range(3, 9)])
}

Se establece el formato de datos de entrada para redes GRU, es decir [observaciones, retardos, caracteristicas]

In [12]:
futuros = 1
pasados  = 12

In [13]:
datosX = []
datosY = []
for i in range(pasados, len(datosNormalizados) - futuros + 1):
  datosX.append(datosNormalizados.iloc[i-pasados:i, 0:datosNormalizados.shape[1]])
  datosY.append(datosNormalizados.iloc[i+futuros-1:i+futuros, 1])


In [14]:
# Convertir las listas en arrays numpy
datosX = np.array(datosX)
datosY = np.array(datosY)

# Ver las dimensiones (shape) de los arrays
print("Dimensiones de X:", datosX.shape)  # (n_muestras, pasos_de_tiempo, n_características)
print("Dimensiones de Y:", datosY.shape)  # (n_muestras, n_características)

Dimensiones de X: (52404, 12, 5)
Dimensiones de Y: (52404, 1)


In [15]:
inputs = datosX.shape[1] * datosX.shape[2]
datosX = datosX.reshape(datosX.shape[0], inputs)

In [16]:
print("Dimensiones de X después de rehape:", datosX.shape)

Dimensiones de X después de rehape: (52404, 60)


Se dividen nuevamente los conjuntos de datos

In [17]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainX, testX = train_test_split(datosX, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testX, valX = train_test_split(testX, test_size=0.33, shuffle=False)

print("Las dimensiones de trainX son: ", trainX.shape)
print("Las dimensiones de testX son: ", testX.shape)
print("Las dimensiones de valX son: ", valX.shape)


Las dimensiones de trainX son:  (36682, 60)
Las dimensiones de testX son:  (10533, 60)
Las dimensiones de valX son:  (5189, 60)


In [18]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainY, testY = train_test_split(datosY, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testY, valY = train_test_split(testY, test_size=0.33, shuffle=False)

print("Las dimensiones de trainY son: ", trainY.shape)
print("Las dimensiones de testY son: ", testY.shape)
print("Las dimensiones de valY son: ", valY.shape)

Las dimensiones de trainY son:  (36682, 1)
Las dimensiones de testY son:  (10533, 1)
Las dimensiones de valY son:  (5189, 1)


Se crean métricas para medir desempeño

In [19]:
import tensorflow.keras.backend as K

def smape(y_true, y_pred):
    """
    Define la función SMAPE (Error Porcentual Absoluto Medio Simétrico).
    """
    summ = K.abs(y_true) + K.abs(y_pred)
    smape_val = K.abs(y_pred - y_true) / summ * 2.0
    return K.mean(smape_val, axis=-1)

def rmse(y_true, y_pred):
    return K.sqrt(K.mean(K.square(y_pred - y_true)))

def ia(y_true, y_pred):
    numerator = K.sum(K.abs(y_true - y_pred))
    denominator = K.sum(K.abs(y_true - K.mean(y_true)) + K.abs(y_pred - K.mean(y_true)))
    return 1 - (numerator / denominator)

In [20]:
from sklearn.metrics import mean_squared_error as mse
from sklearn.metrics import mean_absolute_error as mae

def graficarPrediccion(modelo, x, y, inicio, final):
  predicciones = modelo.predict(x)
  predicciones = predicciones.flatten()
  df = pd.DataFrame({'Originales': y, 'Predichos': predicciones})
  plt.plot(df.index, df['Originales'][inicio:final], label='Originales')
  plt.plot(df.index, df['Predichos'][inicio:final], label='Predichos')
  return df, mse(y, predicciones),  mae(y, predicciones), rmse(y, predicciones), smape(y,predicciones), ia(y, predicciones)

Versión Final


In [21]:
def objective(params):

    model = Sequential()
    model.add(InputLayer(input_shape=(trainX.shape[1],)))
    if (params['layers'] == 1):
      model.add(Dense(units=params['units'], activation=params['activation']))
      model.add(Dropout(params['dropout']))

    else:
      for _ in range(int(params['layers']) - 1):
          model.add(Dense(units=params['units'], activation=params['activation']))
          model.add(Dropout(params['dropout']))
      model.add(Dense(units=params['units'], activation=params['activation']))
      model.add(Dropout(params['dropout']))

    model.add(Dense(1))


    opt = Adam(learning_rate=params['learning_rate'])
    model.compile(optimizer=opt, loss='mse', metrics=["mae", smape, rmse, ia])

    early_stopping = EarlyStopping(monitor='val_loss', patience=15, verbose=1, restore_best_weights= True)

    model.fit(trainX, trainY, epochs=params['epochs'],
                        validation_data=(testX, testY),
                        verbose = 2, batch_size=params['batch'], callbacks=[early_stopping])


    predictions = model.predict(valX)


    loss = mean_squared_error(valY, predictions)

    return {'loss': loss, 'status': STATUS_OK}

In [22]:
trials = Trials()
best = fmin(objective, space, algo=tpe.suggest, max_evals=50, trials=trials, rstate=np.random.default_rng(42))

print(best)

  0%|          | 0/50 [00:00<?, ?trial/s, best loss=?]

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                           

1147/1147 - 9s - 8ms/step - ia: 0.4374 - loss: 1.7772 - mae: 0.9238 - rmse: 1.2108 - smape: 1.2541 - val_ia: 0.4226 - val_loss: 0.2743 - val_mae: 0.4316 - val_rmse: 0.4818 - val_smape: 0.8933

Epoch 2/128                                           

1147/1147 - 3s - 2ms/step - ia: 0.5983 - loss: 0.5611 - mae: 0.5610 - rmse: 0.7380 - smape: 1.0072 - val_ia: 0.5367 - val_loss: 0.1324 - val_mae: 0.2951 - val_rmse: 0.3361 - val_smape: 0.5880

Epoch 3/128                                           

1147/1147 - 3s - 3ms/step - ia: 0.6615 - loss: 0.4114 - mae: 0.4757 - rmse: 0.6334 - smape: 0.8798 - val_ia: 0.5546 - val_loss: 0.1172 - val_mae: 0.2781 - val_rmse: 0.3153 - val_smape: 0.5464

Epoch 4/128                                           

1147/1147 - 7s - 6ms/step - ia: 0.7019 - loss: 0.3322 - mae: 0.4235 - rmse: 0.5689 - smape: 0.7802 - val_ia: 0.6015 - val_loss: 0.0812 - val_mae: 0.2307 - val_rmse: 0.2634 - val_smape: 0.4647

Epoc

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/16                                                                          

144/144 - 7s - 47ms/step - ia: 0.8887 - loss: 0.0827 - mae: 0.1702 - rmse: 0.2299 - smape: 0.4036 - val_ia: 0.9339 - val_loss: 0.0145 - val_mae: 0.0873 - val_rmse: 0.1195 - val_smape: 0.2309

Epoch 2/16                                                                          

144/144 - 1s - 5ms/step - ia: 0.9614 - loss: 0.0083 - mae: 0.0638 - rmse: 0.0900 - smape: 0.1998 - val_ia: 0.9375 - val_loss: 0.0123 - val_mae: 0.0832 - val_rmse: 0.1087 - val_smape: 0.2162

Epoch 3/16                                                                          

144/144 - 1s - 4ms/step - ia: 0.9672 - loss: 0.0061 - mae: 0.0542 - rmse: 0.0772 - smape: 0.1783 - val_ia: 0.9448 - val_loss: 0.0102 - val_mae: 0.0741 - val_rmse: 0.0992 - val_smape: 0.2051

Epoch 4/16                                                                          

144/144 - 1s - 4ms/step - ia: 0.9688 - loss: 0.0054 - mae: 0.0514 - rmse: 0.0730 -

...

Epoch 1/256                                                                            

144/144 - 9s - 63ms/step - ia: 0.6967 - loss: 0.6324 - mae: 0.5467 - rmse: 0.7123 - smape: 0.8259 - val_ia: 0.9060 - val_loss: 0.0290 - val_mae: 0.1253 - val_rmse: 0.1705 - val_smape: 0.3286

Epoch 2/256                                                                            

144/144 - 1s - 6ms/step - ia: 0.8459 - loss: 0.1182 - mae: 0.2568 - rmse: 0.3412 - smape: 0.5381 - val_ia: 0.9126 - val_loss: 0.0228 - val_mae: 0.1172 - val_rmse: 0.1505 - val_smape: 0.3312

Epoch 3/256                                                                            

144/144 - 1s - 7ms/step - ia: 0.8817 - loss: 0.0695 - mae: 0.1954 - rmse: 0.2622 - smape: 0.4362 - val_ia: 0.9275 - val_loss: 0.0161 - val_mae: 0.0965 - val_rmse: 0.1267 - val_smape: 0.2828

Epoch 4/256                                                                            

144/144 - 1s - 10ms/step - ia: 0.9008 - loss: 0.0489 - mae: 0.1632 - r

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/256                                                                            

144/144 - 10s - 68ms/step - ia: 0.7584 - loss: 0.3670 - mae: 0.4220 - rmse: 0.5518 - smape: 0.7165 - val_ia: 0.9112 - val_loss: 0.0231 - val_mae: 0.1185 - val_rmse: 0.1513 - val_smape: 0.3367

Epoch 2/256                                                                            

144/144 - 1s - 6ms/step - ia: 0.8759 - loss: 0.0773 - mae: 0.2055 - rmse: 0.2757 - smape: 0.4592 - val_ia: 0.9395 - val_loss: 0.0119 - val_mae: 0.0804 - val_rmse: 0.1080 - val_smape: 0.2052

Epoch 3/256                                                                            

144/144 - 1s - 6ms/step - ia: 0.9038 - loss: 0.0463 - mae: 0.1586 - rmse: 0.2140 - smape: 0.3687 - val_ia: 0.9435 - val_loss: 0.0097 - val_mae: 0.0747 - val_rmse: 0.0972 - val_smape: 0.2204

Epoch 4/256                                                                            

144/144 - 1s - 10ms/step - ia: 0.9180 - loss: 0.0342 - mae: 0.1349 - 

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/256                                                                            

144/144 - 8s - 59ms/step - ia: 0.8301 - loss: 0.2207 - mae: 0.2934 - rmse: 0.3877 - smape: 0.5371 - val_ia: 0.9359 - val_loss: 0.0128 - val_mae: 0.0852 - val_rmse: 0.1120 - val_smape: 0.2402

Epoch 2/256                                                                            

144/144 - 2s - 15ms/step - ia: 0.9121 - loss: 0.0380 - mae: 0.1440 - rmse: 0.1943 - smape: 0.3246 - val_ia: 0.9501 - val_loss: 0.0084 - val_mae: 0.0661 - val_rmse: 0.0913 - val_smape: 0.1997

Epoch 3/256                                                                            

144/144 - 2s - 12ms/step - ia: 0.9221 - loss: 0.0304 - mae: 0.1274 - rmse: 0.1738 - smape: 0.2863 - val_ia: 0.9422 - val_loss: 0.0101 - val_mae: 0.0762 - val_rmse: 0.0990 - val_smape: 0.2088

Epoch 4/256                                                                            

144/144 - 1s - 10ms/step - ia: 0.9263 - loss: 0.0275 - mae: 0.1204 -

In [23]:
print(best)

{'activation': 3, 'batch': 5, 'dropout': 0.5, 'epochs': 5, 'layers': 1.0, 'learning_rate': 0.005194930724034002, 'units': 3}
